# MoE Unfrozen — `moe_unfrozen_sym_t14_v1` (vast.ai / Jupyter)

Branch: **`hidden_moe_unfrozen`** — warm-start HiDDeN epoch-177, **encoder + decoder backbone train** (no `--freeze-hidden-backbone`).

**Before you start**
1. Rent a GPU instance with **≥50 GB disk** (full COCO) and **≥24 GB VRAM** recommended for unfrozen **batch 128** (lower batch only if OOM).
2. Open Jupyter from the vast.ai console (or SSH port-forward).
3. Set `KAGGLE_USERNAME` / `KAGGLE_KEY` below **or** upload `kaggle.json` in the credentials cell.
4. Push `hidden_moe_unfrozen` to GitHub before cloning, or upload the project folder to `/workspace`.

**Epoch 10 gates:** val `expert_max_use` < 0.50, `train_val_load_l1` < 0.20, noisy BER stable.

## 1. Configuration

In [ ]:
import os
import subprocess
import sys
from glob import glob
from pathlib import Path

# vast.ai often mounts persistent storage at /workspace
WORK_ROOT = os.environ.get("WORKSPACE", "/workspace")
if not Path(WORK_ROOT).exists():
    WORK_ROOT = os.path.expanduser("~")

PROJECT_ROOT = f"{WORK_ROOT}/newmethod"
MOE_DIR = f"{PROJECT_ROOT}/hidden_moe_unfrozen"
DATA_DIR = f"{WORK_ROOT}/coco100k"
EXTRACT_DIR = f"{WORK_ROOT}/coco_extract"
CHECKPOINT_PATH = f"{WORK_ROOT}/checkpoints/my_hidden_experiment--epoch-177.pyt"

REPO_URL = "https://github.com/ademladhari/newmethod.git"
REPO_BRANCH = "main"

KAGGLE_USERNAME = ""  # optional if you upload kaggle.json below
KAGGLE_KEY = ""

EXPERIMENT_NAME = "moe_unfrozen_sym_t14_v1"
BATCH_SIZE = 128  # project default; use 64/32 only if OOM
EPOCHS = 20
NUM_WORKERS = 6
PREFETCH_FACTOR = 3
PRINT_EACH = 100
SAVE_EVERY = 1  # save checkpoint EVERY epoch (required for replication)

# Set True if you already put coco100k on the instance (skips download)
SKIP_COCO_DOWNLOAD = False
LOCAL_COCO_TRAIN = ""  # e.g. /workspace/coco100k/train if pre-uploaded
LOCAL_COCO_VAL = ""

## 2. Install PyTorch + torchvision (CUDA)

Many vast.ai **CUDA-only** images have the driver but **no `torch`**. Run this once per instance.

In [ ]:
import subprocess
import sys

# cu124 works on most vast.ai GPUs (4090, A5000, A6000, A100). Use cu118 only if install fails.
TORCH_CUDA = "cu124"  # or "cu121" / "cu118"

try:
    import torch
    print("torch already installed:", torch.__version__, "| cuda:", torch.version.cuda)
except ModuleNotFoundError:
    print("Installing PyTorch ({}) — ~2–3 min...".format(TORCH_CUDA))
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "torch",
            "torchvision",
            "--index-url",
            "https://download.pytorch.org/whl/{}".format(TORCH_CUDA),
        ]
    )
    import torch
    print("Installed:", torch.__version__, "| cuda:", torch.version.cuda)

assert torch.cuda.is_available(), (
    "PyTorch installed but CUDA not visible. Run: nvidia-smi. "
    "If driver OK, try TORCH_CUDA='cu121' or 'cu118' and re-run this cell."
)
print("GPU:", torch.cuda.get_device_name(0))

## 3. GPU + disk check

In [ ]:
import shutil
import subprocess

import torch  # installed in previous cell

assert torch.cuda.is_available(), "CUDA not available — run cell 2 (Install PyTorch) first"
n_gpu = torch.cuda.device_count()
print("GPUs:", n_gpu)
for i in range(n_gpu):
    print(f"  [{i}]", torch.cuda.get_device_name(i))
print("PyTorch:", torch.__version__)
print("WORK_ROOT:", WORK_ROOT)
print("Disk free:", shutil.disk_usage(WORK_ROOT).free // (1024**3), "GiB")
subprocess.run(["nvidia-smi"], check=False)

## 3. Kaggle API (COCO + checkpoint)

In [ ]:
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json = kaggle_dir / "kaggle.json"

if KAGGLE_USERNAME and KAGGLE_KEY:
    kaggle_json.write_text(
        '{{"username":"{}","key":"{}"}}\n'.format(KAGGLE_USERNAME, KAGGLE_KEY)
    )
    os.chmod(kaggle_json, 0o600)
    print("Wrote", kaggle_json)
elif not kaggle_json.is_file():
    print("Upload kaggle.json to", kaggle_dir)
    try:
        from IPython.display import display
        import ipywidgets as widgets
        upload = widgets.FileUpload(accept=".json", multiple=False)
        display(upload)
        # run next line after upload: kaggle_json.write_bytes(upload.data[0]); os.chmod(kaggle_json, 0o600)
    except Exception:
        print("Or: echo '{\"username\":\"...\",\"key\":\"...\"}' > ~/.kaggle/kaggle.json && chmod 600 ~/.kaggle/kaggle.json")

assert kaggle_json.is_file(), "Missing ~/.kaggle/kaggle.json"

## 4. Clone repo + install deps

In [ ]:
import sys

if Path(PROJECT_ROOT).is_dir():
    print("Repo exists:", PROJECT_ROOT)
else:
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, PROJECT_ROOT],
        check=True,
    )

assert Path(MOE_DIR).is_dir(), f"Missing {MOE_DIR} — push hidden_moe_unfrozen to GitHub or copy folder to WORK_ROOT"

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "kagglehub==0.3.12", "kaggle", "torchvision"],
    check=True,
)
os.chdir(MOE_DIR)
sys.path.insert(0, MOE_DIR)
print("cwd:", os.getcwd())

## 5. COCO → `train/` and `val/` (symlink, no move)

In [ ]:
import zipfile

train_dst = Path(DATA_DIR) / "train"
val_dst = Path(DATA_DIR) / "val"
marker = Path(DATA_DIR) / ".ready"


def count_jpg(folder: Path) -> int:
    if not (folder.is_dir() or folder.is_symlink()):
        return 0
    return sum(1 for f in folder.iterdir() if f.suffix.lower() == ".jpg")


def find_image_dirs(root: Path):
    train_dir = val_dir = None
    for dirpath, _, filenames in os.walk(root):
        if not filenames:
            continue
        base = Path(dirpath).name.lower()
        if base == "train2017" and any(f.lower().endswith(".jpg") for f in filenames):
            train_dir = Path(dirpath)
        if base in ("val2017", "valid2017") and any(f.lower().endswith(".jpg") for f in filenames):
            val_dir = Path(dirpath)
    return train_dir, val_dir


def link_coco(train_src: Path, val_src: Path):
    Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
    for dst in (train_dst, val_dst):
        if dst.is_symlink():
            dst.unlink()
        elif dst.is_dir():
            shutil.rmtree(dst)
    os.symlink(train_src.resolve(), train_dst, target_is_directory=True)
    os.symlink(val_src.resolve(), val_dst, target_is_directory=True)
    marker.write_text("ok\n")


if LOCAL_COCO_TRAIN and LOCAL_COCO_VAL:
    link_coco(Path(LOCAL_COCO_TRAIN), Path(LOCAL_COCO_VAL))
    print("Linked local COCO")
elif SKIP_COCO_DOWNLOAD and marker.is_file():
    print("Using existing", DATA_DIR)
elif count_jpg(train_dst) > 100000 and count_jpg(val_dst) > 4000:
    print("COCO already at", DATA_DIR)
else:
    import kagglehub

    cache = Path(kagglehub.dataset_download("awsaf49/coco-2017-dataset"))
    print("kagglehub cache:", cache)
    train_src, val_src = find_image_dirs(cache)
    if train_src is None or val_src is None:
        archive = next(cache.rglob("*.archive"), None)
        assert archive, "No .archive in cache"
        extract_root = Path(EXTRACT_DIR)
        done = extract_root / ".extract_done"
        if not done.is_file():
            extract_root.mkdir(parents=True, exist_ok=True)
            print("Extracting zip to", extract_root, "(~15 min)...")
            with zipfile.ZipFile(archive, "r") as zf:
                zf.extractall(extract_root)
            done.write_text("ok\n")
        train_src, val_src = find_image_dirs(extract_root)
    assert train_src and val_src
    link_coco(train_src, val_src)

print("train images:", count_jpg(train_dst))
print("val images:", count_jpg(val_dst))

## 6. HiDDeN epoch-177 checkpoint

In [ ]:
ckpt = Path(CHECKPOINT_PATH)
ckpt.parent.mkdir(parents=True, exist_ok=True)

if ckpt.is_file() and ckpt.stat().st_size > 5_000_000:
    print("Checkpoint OK:", ckpt)
else:
    import kagglehub

    cache = Path(kagglehub.dataset_download("wings2ofice2/hiddencheckpointc"))
    matches = list(cache.rglob("*epoch-177.pyt")) or list(cache.rglob("*.pyt"))
    src = max(matches, key=lambda p: p.stat().st_size)
    shutil.copy2(src, ckpt)
    print("Saved:", ckpt, round(ckpt.stat().st_size / 1e6, 2), "MB")

## 7. Train (unfrozen backbone)

In [ ]:
os.chdir(MOE_DIR)

train_cmd = [
    sys.executable, "-u", "train_moe.py", "new",
    "--data-dir", DATA_DIR,
    "--name", EXPERIMENT_NAME,
    "--batch-size", str(BATCH_SIZE),
    "--epochs", str(EPOCHS),
    "--num-experts", "4",
    "--top-k", "1",
    "--balance-loss-weight", "0.04",
    "--balance-loss-start-weight", "0.005",
    "--balance-loss-warmup-epochs", "10",
    "--router-jitter-noise", "0.0",
    "--router-input-dropout", "0.0",
    "--expert-dropout", "0.0",
    "--router-z-loss-weight", "0.001",
    "--router-temperature-start", "1.4",
    "--router-temperature-end", "1.0",
    "--adversarial-loss", "0.0001",
    "--init-hidden-checkpoint", CHECKPOINT_PATH,
    "--enable-fp16",
    "--router-grad-clip-norm", "0.5",
    "--num-workers", str(NUM_WORKERS),
    "--pin-memory",
    "--prefetch-factor", str(PREFETCH_FACTOR),
    "--save-every", str(SAVE_EVERY),
    "--print-each", str(PRINT_EACH),
]
# No --freeze-hidden-backbone → full backbone + MoE train
print(" ".join(train_cmd))
subprocess.run(train_cmd, check=True, cwd=MOE_DIR)

## 8. Tail log while training (optional — run in a second notebook tab)

In [ ]:
from glob import glob
import subprocess

logs = sorted(glob(f"{MOE_DIR}/runs/{EXPERIMENT_NAME}*/*.log"))
if logs:
    print("Log:", logs[-1])
    subprocess.run(["tail", "-n", "40", logs[-1]])
else:
    print("No log yet under runs/")

## 9. Continue after interrupt (optional)

In [ ]:
from glob import glob
import subprocess
import sys

runs = sorted(glob(f"{MOE_DIR}/runs/{EXPERIMENT_NAME}*"))
assert runs, "No run folder found"
RUN_FOLDER = runs[-1]
print("Continue:", RUN_FOLDER)

cont_cmd = [
    sys.executable, "-u", "train_moe.py", "continue",
    "--folder", RUN_FOLDER,
    "--data-dir", DATA_DIR,
    "--epochs", str(EPOCHS),
    "--num-workers", str(NUM_WORKERS),
    "--pin-memory",
    "--prefetch-factor", str(PREFETCH_FACTOR),
    "--save-every", str(SAVE_EVERY),
    "--print-each", str(PRINT_EACH),
]
subprocess.run(cont_cmd, check=True, cwd=MOE_DIR)

## 10. Pack results for download

In [ ]:
from glob import glob
import subprocess
from pathlib import Path

runs = sorted(glob(f"{MOE_DIR}/runs/{EXPERIMENT_NAME}*"))
if not runs:
    print("No run to zip")
else:
    latest = runs[-1]
    zip_out = f"{WORK_ROOT}/{EXPERIMENT_NAME}_run.zip"
    subprocess.run(
        ["zip", "-r", zip_out, os.path.basename(latest)],
        check=True,
        cwd=f"{MOE_DIR}/runs",
    )
    print("Created:", zip_out, round(Path(zip_out).stat().st_size / 1e6, 1), "MB")
    for name in ["train.csv", "validation.csv", "validation_noisy.csv"]:
        p = Path(latest) / name
        if p.is_file():
            print("\n===", name, "last 2 epochs ===")
            subprocess.run(["tail", "-n", "3", str(p)])